In [1]:
! pip install yfinance

In [2]:
import pandas as pd
import numpy as np

In [3]:
import yfinance as yf

df = yf.download("^NSEI", start='2018-01-01', end='2024-01-01')

# yfinance >= 0.2 returns a MultiIndex DataFrame; flatten to simple column names
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df.index.name = 'Date'
df = df.reset_index()

[*********************100%***********************]  1 of 1 completed


In [4]:
df.head()

,Date,Close,High,Low,Open,Volume
0,2018-01-02,10442.200195,10495.200195,10404.650391,10477.549805,153400
1,2018-01-03,10443.200195,10503.599609,10429.549805,10482.650391,167300
2,2018-01-04,10504.799805,10513.000000,10441.450195,10469.400391,174900
3,2018-01-05,10558.849609,10566.099609,10520.099609,10534.250000,180900
4,2018-01-08,10623.599609,10631.200195,10588.549805,10591.700195,169000


In [5]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "nifty50_raw.csv"
if not csv_path.exists():
    df.to_csv(csv_path, index=False)

df = pd.read_csv(csv_path)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1477 entries, 0 to 1476
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    1477 non-null   str    
 1   Close   1477 non-null   float64
 2   High    1477 non-null   float64
 3   Low     1477 non-null   float64
 4   Open    1477 non-null   float64
 5   Volume  1477 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 69.4 KB


In [7]:
df.isnull().sum()

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [8]:
# Feature engineering
df['MA20'] = df['Close'].rolling(window=20).mean()
df['MA50'] = df['Close'].rolling(window=50).mean()

delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = -delta.where(delta < 0, 0).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / loss))

df['BB_upper'] = df['MA20'] + 2 * df['Close'].rolling(20).std()
df['BB_lower'] = df['MA20'] - 2 * df['Close'].rolling(20).std()

# Optional momentum feature
df['Returns'] = df['Close'].pct_change()

In [9]:
from sklearn.preprocessing import MinMaxScaler

feature_cols = [
    'Close', 'Open', 'High', 'Low', 'Volume',
    'MA20', 'MA50', 'RSI', 'BB_upper', 'BB_lower', 'Returns'
 ]

# Drop warm-up rows created by rolling indicators
df_model = df[feature_cols].dropna().copy()

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

features_scaled = feature_scaler.fit_transform(df_model[feature_cols])
target_scaled = target_scaler.fit_transform(df_model[['Close']]).reshape(-1)

In [10]:
# Create multi-feature sequences for LSTM
def create_sequences(features, target, window=60):
    X, y = [], []
    for i in range(window, len(features)):
        X.append(features[i-window:i])
        y.append(target[i])
    return np.array(X), np.array(y)

X, y = create_sequences(features_scaled, target_scaled, window=60)

In [11]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# X is already shaped as (samples, timesteps, features)
print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('Features used:', len(feature_cols))

X_train shape: (1094, 60, 11)
X_test shape : (274, 60, 11)
Features used: 11


In [12]:
import pickle
with open('data/processed_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_scaler': feature_scaler,
        'target_scaler': target_scaler,
        'feature_cols': feature_cols
    }, f)
print('Saved multi-feature processed data')

Saved multi-feature processed data


In [14]:
# Optional quick check of saved payload
import pickle
with open('data/processed_data.pkl', 'rb') as f:
    payload = pickle.load(f)

print('Saved keys:', list(payload.keys()) if isinstance(payload, dict) else 'legacy tuple format')

EOFError: Ran out of input